# Lab 04: Functions, Batch Reports, and Pseudocode
Introduction to Programming | Space Engineering | 2026/2027  
Name: *write here*  
Date: *write here*

Your Lab 03 monitor can classify a sequence of readings. Today you will organize that logic into functions, reuse it in a batch processor, and check that a plausible result actually meets its specification. The home task extends this into a report with input validation.

By the end, aim to:
- define functions with parameters and useful return values;
- combine functions, loops, and decision rules without copying the classifier;
- distinguish displaying a value from returning it;
- find logical errors involving hidden inputs and early returns;
- turn written requirements into pseudocode before writing Python.

## Route

| Portion | Activities | Planning time |
|---|---|---:|
| Start | Open the notebook and course kernel | 5 minutes |
| Part A | A1–A4: functions, repairs, batch processing, summaries | 65 minutes |
| End of class | Pseudocode introduction | 5 minutes |
| End of class | Individual paper exercise | 10 minutes |
| Checkpoint | Save, commit, push | 5 minutes |
| Part B, at home | B1–B4: validate, extend, test, audit | 75–100 minutes |

These are estimates. Keep the final 10 minute paper exercise even if we agree on an earlier coding stopping point. Use the optional extension only if there is spare time.

Use the existing course environment and start/finish checklist. No new packages, data downloads, or external files are needed. All measurements and rules here are simplified teaching examples.

In class, make your first diagnosis and attempt yourself before asking an LLM. At home you may use one, but record a meaningful prompt and the evidence used to check the response. An LLM can help implement your rules; you still need to specify what those rules mean.

Repair marked code in place. For each repair, record one failing input, its expected and actual results, and the cause. Some supplied checks should fail before the repairs are complete. Do not change an expected result just to make a test pass.

## Part A: in class
### A1. Give repeated work a name | 12 minutes

A function groups a task under a name. Parameters are the named inputs in its definition; arguments are the actual values supplied when you call it. A definition does not carry out the task until the function is called.

```python
def function_name(parameter):
    # Work with the input.
    return result
```

`return` sends a value to the caller and immediately ends that call. `print()` displays something; it does not send that displayed value back as the result. A function that reaches its end without `return` produces `None`, meaning no result value here.

The complete example below estimates recording size from a constant data rate. Before running, predict the three final values. Which changes when `rate_mb_s` is reassigned, and which has already been calculated?

In [ ]:
def estimate_recording_mb(duration_s, rate_mb_s):
    size_mb = duration_s * rate_mb_s
    return size_mb

rate_mb_s = 2
first_size_mb = estimate_recording_mb(10, rate_mb_s)
rate_mb_s = 3
second_size_mb = estimate_recording_mb(10, rate_mb_s)

print("First:", first_size_mb, "MB")
print("Second:", second_size_mb, "MB")
print("Combined:", first_size_mb + second_size_mb, "MB")

My prediction and explanation:

### Change the function's input, not its definition

Write two calls in the next cell: a 15 second recording at 4 MB/s, then a recording of zero seconds at the same rate. Store the returned values and print them. Do not type the calculated sizes into your program.

Then answer: if the example function used `print(size_mb)` instead of `return size_mb`, what would `first_size_mb` contain? Why would adding the two results fail?

In [ ]:
# Call estimate_recording_mb twice, store the results, and display them.

### Inputs a caller can see

Names assigned inside a function, including its parameters, are local to that call. The function can also read names defined outside it, but that can hide a dependency from the caller. In today's functions, use the parameters for measurements, not a remembered notebook variable.

If you edit a function definition, run its cell again before testing it. Python otherwise continues using the earlier definition. Changing a parameter name in the definition also means changing its uses in the body.

### A2. Audit a reusable power classifier | 18 minutes

Use these familiar battery and voltage rules. Select the first applicable status:

1. CRITICAL if battery is below 15% or voltage is below 22 V.
2. SAFE MODE if battery is below 25% or voltage is below 24 V.
3. WARNING if battery is at most 35%.
4. NOMINAL otherwise.

For now, all inputs are valid numeric measurements. The function must return exactly one status string for each call, without printing. Its answer must depend only on its two arguments.

The supplied function runs but has three distinct logical problems. Use the four calls below it to investigate, then repair it. The outside variable is deliberate: a correct function must not depend on it.

In [ ]:
bus_voltage_v = 28

def classify_power(battery_percent, voltage_v):
    if battery_percent < 25 or bus_voltage_v < 24:
        return "SAFE MODE"
    elif battery_percent < 15 or bus_voltage_v < 22:
        return "CRITICAL"
    elif battery_percent <= 35:
        return "WARNING"
    else:
        print("NOMINAL")

print("Healthy:", classify_power(80, 28))
print("Low battery:", classify_power(10, 28))
print("Low voltage:", classify_power(80, 21))
print("Warning boundary:", classify_power(35, 28))

| Problem | Call that exposes it | Required returned value | Faulty returned value or extra output | My repair |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |

After repairing:
- Change the outside `bus_voltage_v` to 20 and repeat `classify_power(80, 28)`. Its answer should not change. Restore the outside variable to 28 afterward.
- Test battery values 14, 15, and 16 with voltage fixed at 28 V. Explain the status exactly at 15%.
- Test voltages 21, 22, and 23 with battery fixed at 80%. Explain the status exactly at 22 V.

Only one branch of a call needs to return. Multiple `return` lines are appropriate when different branches produce different answers.

Stretch: after the repairs, would replacing the `elif` keywords with separate `if` statements change the returned result here? Explain how an immediate `return` differs from assigning `status` and continuing through later statements.

### A3. Classify the whole batch | 20 minutes

Printing statuses makes them visible, but returning a list lets another part of the program use them.

`results = []` creates an empty list. `results.append(value)` adds one value to its end. To add a label and a status as one record, use `results.append((label, status))`. The parentheses group them into a tuple, as in Lab 03.

Use `.append(...)` on its own line. It changes the list and returns `None`; `results = results.append(...)` would replace your list with `None`.

Requirement for `classify_batch(records)`:
- Input: a list of `(label, battery_percent, voltage_v)` tuples, in observation order.
- Call `classify_power` for every record; do not copy its decision chain.
- Return a list of `(label, status)` tuples in the same order.
- Keep repeated records, even if their labels are identical.
- For an empty input, return `[]`.
- Every call must start a fresh result list. Leave the input list unchanged and do not print inside the function.

The following code returns a plausible list, but misses most of the batch. Trace how far one call gets before returning. Repair it and explain why your repair also handles empty input.

In [ ]:
def classify_batch(records):
    for label, battery_percent, voltage_v in records:
        status = classify_power(battery_percent, voltage_v)
        results.append((label, status))
        return results
        
results = []
power_records = [
    ("P1", 80, 28),
    ("P2", 10, 28),
    ("P3", 80, 21),
    ("P4", 15, 28),
    ("P5", 35, 28),
    ("P1", 80, 28),
]

print(classify_batch(power_records))
print("Empty batch:", classify_batch([]))

My explanation of the early return:

Why the empty input needs a return after the loop:

### Check behavior across calls

Predict what the following checks will test, then run them after the repairs. `assert condition` raises `AssertionError` if the condition is false. A message after the comma describes the failed expectation. Passing a check supplies evidence for that case; it does not establish that every possible case is correct.

Here, `len(a_list)` gives the number of elements in the list. A single tuple inside a list counts as one element.

In [ ]:
expected_power_results = [
    ("P1", "NOMINAL"),
    ("P2", "CRITICAL"),
    ("P3", "CRITICAL"),
    ("P4", "SAFE MODE"),
    ("P5", "WARNING"),
    ("P1", "NOMINAL"),
]

assert classify_batch(power_records) == expected_power_results, "Check statuses and order"
assert classify_batch([]) == [], "An empty batch must return an empty list"
assert len(power_records) == 6, "Do not remove input records while processing"
print("Supplied batch checks passed")

Add a test of your own. Use at least two records and state which specific mistake it catches. A test with the records in reverse order can reveal an order or hidden-state problem.

Input and expected result:  
Mistake this would expose:

In [ ]:
# Add one call and an assert with your independently predicted result.

### A4. Return a useful summary | 15 minutes

Build one more function, `summarize_power(classified_records)`. Its input is the returned list of `(label, status)` tuples, not the original measurements. Reuse your Lab 03 counter pattern inside it.

Return four counts as one tuple, in this fixed order:
`(critical_count, safe_count, warning_count, nominal_count)`.

For example, `return critical_count, safe_count, warning_count, nominal_count` groups the four values into a tuple. The caller can unpack it with four names, just as a loop can unpack a record.

Requirements:
- Count every record once, including repeated labels.
- An empty input returns `(0, 0, 0, 0)`.
- Do not change the input list or print inside the function.
- All input statuses are one of the four documented strings. You do not need to handle other strings here.

First write the expected four counts for `power_records`: ______.
Then implement the function. The placeholder returns zeros so the cell is valid Python; it is not a completed solution.

In [ ]:
def summarize_power(classified_records):
    # Initialize four counters, loop through the records, and update one count each time.
    # Replace the placeholder return with the computed counts.
    return 0, 0, 0, 0

classified_records = classify_batch(power_records)
critical, safe, warning, nominal = summarize_power(classified_records)
print("Critical:", critical, "Safe:", safe, "Warning:", warning, "Nominal:", nominal)

In [ ]:
assert summarize_power(classify_batch(power_records)) == (2, 1, 1, 2)
assert summarize_power([]) == (0, 0, 0, 0)
assert summarize_power([("same", "WARNING"), ("same", "WARNING")]) == (0, 0, 2, 0)
print("Supplied summary checks passed")

Explain in 1–2 sentences each:
1. Why does the batch processor call `classify_power` rather than repeat its conditions?
2. Why does the summary consume classified records rather than calculate the statuses again?
3. If the four counts add up correctly, what could still be wrong?

Stretch: change one boundary rule in a temporary copy. Which tests detect it, and which still pass? Restore the required rule and rerun the definition afterward.

## Before the paper exercise: what is pseudocode? | 5 minutes

Pseudocode is a clear description of an algorithm using ordinary words and programming ideas. You decide the steps, their order, and what information must be remembered before dealing with Python syntax.

There is no single required notation. Use meaningful names, indentation, and words such as `SET`, `FOR EACH`, `IF`, `ELSE`, and `RETURN`. You can use comparisons and arithmetic. You do not need colons, exact Python function syntax, or a particular spelling of a keyword.

For example, a function that counts temperatures above a supplied limit could be planned as:

```text
FUNCTION count_hot_readings(temperatures, limit)
    SET hot_count TO 0
    FOR EACH temperature IN temperatures
        IF temperature > limit
            SET hot_count TO hot_count + 1
    RETURN hot_count
```

A useful plan makes these things explicit:
- What enters the algorithm, and what must come out?
- What values are initialized before repetition begins?
- What happens for each item, and when do remembered values change?
- Which branches are possible, and when is the final result returned?

“Process the data and return the answer” is too vague to implement or test. A detailed plan is also useful context for an LLM: it makes your intended rules visible. You still need to check the implementation.

## Individual paper exercise: choose files for a downlink | 10 minutes

Use pen and paper, without running Python, using an LLM, or discussing answers. Write your name on the sheet. Focus on an unambiguous algorithm; Python syntax is not required. You do not need to implement this task during class.

A satellite has a fixed amount of free downlink capacity, measured in MB. Files are given in order as `(label, size_mb, priority)` records. Sizes are positive numbers; capacity is zero or greater. Priority is either `HIGH` or `LOW`.

Design `choose_downlink(files, capacity_mb)` using these rules:
1. Inspect each file once, in the supplied order. Do not sort.
2. Select a file only if its priority is HIGH and its size is no greater than the remaining capacity.
3. When a file is selected, record its label and immediately reduce remaining capacity by its size.
4. Otherwise skip that file and keep checking later files. A file that is too large does not end the search.
5. Return the selected labels in order and the remaining capacity. For an empty list, return an empty selection and the original capacity.

Your tasks:
- About 6 minutes: write pseudocode for the complete function, including initialization and return.
- About 3 minutes: trace your algorithm for the data below. Show each file's decision and the remaining capacity after that decision.
- About 1 minute: state the result for an empty list with capacity 10 MB, and name one mistake that your trace would catch.

Starting capacity: 10 MB.

| Label | Size, MB | Priority |
|---|---:|---|
| A | 6 | HIGH |
| B | 3 | LOW |
| C | 7 | HIGH |
| D | 4 | HIGH |
| E | 1 | HIGH |

The exercise checks your choice of steps, correct placement of the capacity update, continued searching after a skipped file, and the final result. Exact keyword style is not assessed. Hand in the sheet as directed before using a computer again.

## Class checkpoint | 5 minutes

- [ ] I completed the paper exercise independently.
- [ ] I repaired the required class functions and recorded the reasons.
- [ ] I reran the function definitions after editing them, then checked their callers.
- [ ] I restarted the kernel and ran the completed Part A cells in order.
- [ ] I saved, committed, and pushed my class work using the course checklist.

Use a commit message such as `Complete Lab 04 class work`. Unfinished home functions and their checks are separate from this checkpoint. Save the home work in a later commit.

If we shorten the class coding route, record the agreed stopping point here: ______. The home assignment does not automatically expand to include unfinished class tasks.

## Part B: at home
### B1. Extend the health monitor without duplicating it

Bring together your Lab 03 monitor and today's function design. Write a function named:

```python
classify_health(battery_percent, voltage_v, temperature_c, comms_ok)
```

It must return one status string and must not print or depend on notebook measurements outside the function.

This version adds validation. Inputs are always numbers for the three measurements and a Boolean for communications; there are no strings, missing values, or NaN values to handle. Use these rules in order:

1. INVALID if battery is outside the inclusive interval 0–100%, or voltage is outside the inclusive interval 0–40 V. This validation takes priority over every operational status. An invalid record contributes to INVALID only.
2. CRITICAL if battery is below 15% or voltage is below 22 V.
3. SAFE MODE if battery is below 25%, voltage is below 24 V, temperature is outside the inclusive interval −20°C to 85°C, or communications are unavailable.
4. WARNING if battery is at most 35% or temperature is above 75°C.
5. NOMINAL otherwise.

Do not treat an extreme temperature as INVALID: its specified outcome is SAFE MODE unless a higher priority rule applies. Unavailable communications does not make a record INVALID either. These voltage validity limits are teaching assumptions, not engineering guidance.

You can reuse your Lab 03 decision chain, but verify it against this full specification. Keep `classify_power` unchanged so that Part A still works.

Before implementing, fill in the expected statuses. Give reasons for H and I, where validation competes with a fault, and L, where the temperature is unusual but the input is not INVALID.

| Label | Battery % | Voltage V | Temperature °C | Comms | Expected status |
|---|---:|---:|---:|---|---|
| A | 80 | 28 | 20 | True | |
| B | 10 | 28 | 20 | True | |
| C | 80 | 21 | 20 | True | |
| D | 15 | 24 | −20 | True | |
| E | 25 | 24 | 85 | True | |
| F | 35 | 28 | 76 | False | |
| G | 36 | 28 | 76 | True | |
| H | −1 | 28 | 20 | True | |
| I | 10 | 41 | 20 | True | |
| J | 100 | 40 | 75 | True | |
| K | 80 | 28 | 20 | True | |
| L | 80 | 28 | 120 | True | |

H:  
I:  
L:

In [ ]:
def classify_health(battery_percent, voltage_v, temperature_c, comms_ok):
    # Implement validation first, then the operational priority rules.
    return "NOT IMPLEMENTED"

### B2. Build a batch report from functions

Implement two functions with the interfaces below. Follow the Part A patterns; the extra input fields and INVALID state are the changes.

`classify_health_batch(records)`:
- Accept a list of `(label, battery_percent, voltage_v, temperature_c, comms_ok)` tuples.
- Call `classify_health` exactly once for each record.
- Return `(label, status)` tuples in a new list, preserving order and repeated records.
- Return `[]` for empty input and leave the input unchanged.

`summarize_health(classified_records)`:
- Accept the resulting list of `(label, status)` tuples. All statuses are one of the five required strings.
- Return `(invalid_count, critical_count, safe_count, warning_count, nominal_count)`.
- Return five zeros for empty input. Do not reclassify measurements.

Neither function should print or use a result list or counters stored outside the function. Display the report in the separate caller cell after the definitions. You do not need dictionaries, file input, or extra packages.

In [ ]:
def classify_health_batch(records):
    # Create a fresh list, classify each record, append its result, then return the list.
    return []

def summarize_health(classified_records):
    # Count the five categories and return them in the documented order.
    return 0, 0, 0, 0, 0

In [ ]:
health_records = [
    ("A", 80, 28, 20, True),
    ("B", 10, 28, 20, True),
    ("C", 80, 21, 20, True),
    ("D", 15, 24, -20, True),
    ("E", 25, 24, 85, True),
    ("F", 35, 28, 76, False),
    ("G", 36, 28, 76, True),
    ("H", -1, 28, 20, True),
    ("I", 10, 41, 20, True),
    ("J", 100, 40, 75, True),
    ("K", 80, 28, 20, True),
    ("L", 80, 28, 120, True),
]

health_results = classify_health_batch(health_records)
for label, status in health_results:
    print(label, status)

invalid, critical, safe, warning, nominal = summarize_health(health_results)
print("Invalid:", invalid)
print("Critical:", critical)
print("Safe mode:", safe)
print("Warning:", warning)
print("Nominal:", nominal)
print("Total classified:", len(health_results))

### B3. Tests that distinguish implementations

Start with the supplied checks. They should fail until the relevant functions are completed. If a check fails, identify the earliest wrong intermediate result: the individual classification, the batch list, or the summary. Do not rewrite the same classifier inside your tests.

Then turn your predicted A–L statuses into one expected list and compare it with the full batch result. Independently predict the five summary counts and check those too. Complete the response table below before running your added tests.

In [ ]:
assert classify_health(-1, 28, 20, True) == "INVALID"
assert classify_health(10, 41, 20, False) == "INVALID", "Validation has first priority"
assert classify_health(0, 28, 20, True) == "CRITICAL", "Zero charge is valid but critical"
assert classify_health(100, 40, 75, True) == "NOMINAL", "Upper validity endpoints are included"
assert classify_health(80, 28, 120, True) == "SAFE MODE", "Temperature is not an INVALID rule"

assert classify_health_batch([]) == []
assert summarize_health([]) == (0, 0, 0, 0, 0)

small_batch = [("X", 10, 28, 20, True), ("Y", 80, 28, 20, True)]
expected_small = [("X", "CRITICAL"), ("Y", "NOMINAL")]
assert classify_health_batch(small_batch) == expected_small
assert classify_health_batch(small_batch) == expected_small, "A second call must start fresh"
assert summarize_health(expected_small) == (0, 1, 0, 0, 1)
print("Supplied home checks passed")

Add tests covering these four groups. Use short batches rather than changing function definitions for each case.

1. The supplied A–L list: every status, their order, and all five counts.
2. Three temperatures: −21°C, −20°C, and −19°C, with battery 80%, voltage 28 V, and communications available. Also test 84°C, 85°C, and 86°C. Remember the WARNING rule applies above 75°C even while the temperature is within the safe interval.
3. Repeated labels and reversed input order. For example, two copies of a critical record must both be counted; a nominal record after them must remain nominal.
4. One test you design to expose a mistake not isolated by the supplied checks. Name the faulty behavior you are testing.

For each group, write the expected results first. The number of returned records should equal the input length, and the five counts should add up to that length. Also compare individual statuses: totals alone can hide misclassification.

| Test group | Exact input or labels | Expected statuses and counts | Observed result | What this test can catch |
|---|---|---|---|---|
| A–L | | | | |
| Temperature boundaries | | | | |
| Repeats and order | | | | |
| My additional case | | | | |

In [ ]:
# Add assertions for your predicted full result list, summary counts, and other test groups.
# Keep the tests here so that a clean run repeats your verification.

### B4. Audit a change request before accepting code

A colleague asks an LLM:

> Make the monitor ignore bad readings and give me the counts.

It proposes this fragment, where `records` contains five-field health records:

```python
results = []
for label, battery, voltage, temperature, comms in records:
    if 0 < battery < 100 and 0 < voltage < 40:
        results.append((label, classify_health(battery, voltage, temperature, comms)))
return results
```

Treat this as a fragment inside a function, so its final `return` is syntactically valid. Judge it against our B1–B2 specification.

In a short audit:
1. Give a valid boundary record that this fragment silently removes. State its required status.
2. Give an invalid record that our report must retain as INVALID. Explain how silently dropping it can make a report misleading, even if the remaining counts look reasonable.
3. Write a replacement prompt explaining how invalid records, boundary values, output order, and category counts must be handled. Include two concrete examples with expected outputs.

You do not need an LLM account to complete this task. If you use an LLM elsewhere, record one relevant prompt, summarize what you accepted or changed, and cite an exact test result that checked it. Otherwise state that you used the supplied audit only.

Valid boundary record, required status, and why it is lost:

Invalid record and why retaining it matters:

My replacement prompt:
> Write here.

Assistance used and evidence checked:

One remaining limitation of my tests:

## Home submission checklist

- [ ] Each classifier returns a value based on its parameters, with no hidden measurement inputs.
- [ ] Validation precedes the operational rules in `classify_health`.
- [ ] Batch functions retain every record, including invalid or repeated records.
- [ ] Summaries count returned statuses without running the classifier again.
- [ ] I recorded predictions and added executable tests for B3.
- [ ] I completed the audit and assistance record.
- [ ] I restarted the kernel and ran the notebook in order; every required self-check passed.
- [ ] I saved, committed, and pushed the home work separately from the class checkpoint.

Be prepared to explain a function call, a return value, and one test that distinguishes your implementation from a plausible incorrect one.

## Optional extension: turn your paper plan into Python

After the paper exercise has been handed in, implement `choose_downlink(files, capacity_mb)` using the same specification. Keep the original paper plan; if you change the algorithm, explain why.

Return `(selected_labels, remaining_capacity_mb)`. Test the supplied example, an empty list, zero capacity, a file that exactly fills capacity, and a file too large followed by one that fits. Use assertions with independently predicted answers.

Explain why selecting a file must update capacity inside the loop, while the final return belongs after the loop. This extension is not part of the required home work.

In [ ]:
# Optional: define and test choose_downlink(files, capacity_mb).